# THEMIS-Aの電磁場データについて、full orbit (8 Hz)とparticle burst (512 Hz)、low telemetry (16 Hz)とhigh telemetry (128 Hz)の両方を用いる。

# データ保存先

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# THEMIS-Aの電場・磁場データのdownload

In [ ]:
import pyspedas as psp
import pytplot as pt

pt.del_data('*')

time_range = ['20220901/20:00:00', '20220902/00:00:00']

path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/2230-2330'

psp.themis.fgm(trange=time_range, probe='a', level='l2', no_update=True)     # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp', no_update=True)    # efp: 512 Hz
psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efi', no_update=True)                    # eff: 8 Hz

print("--- Loaded tplot variables ---")
print(pt.tplot_names())

In [ ]:
E8_data_gsm     = pt.data_quants['tha_eff_dot0_gsm']
E512_data_gsm   = pt.data_quants['tha_efp_gsm']
Bspin_data_gsm  = pt.data_quants['tha_fgs_gsm']
B16_data_gsm    = pt.data_quants['tha_fgl_gsm']
B128_data_gsm   = pt.data_quants['tha_fgh_gsm']

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E8_data_gsm     = E8_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E512_data_gsm   = E512_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
Bspin_data_gsm  = Bspin_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B16_data_gsm    = B16_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B128_data_gsm   = B128_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

In [ ]:
import numpy as np

print((Bspin_data_gsm.time[1] - Bspin_data_gsm.time[0]) / np.timedelta64(1, 's'))
print(1/((B16_data_gsm.time[1] - B16_data_gsm.time[0]) / np.timedelta64(1, 's')))
print(1/((B128_data_gsm.time[1] - B128_data_gsm.time[0]) / np.timedelta64(1, 's')))
print(1/((E8_data_gsm.time[1] - E8_data_gsm.time[0]) / np.timedelta64(1, 's')))
print(1/((E512_data_gsm.time[1] - E512_data_gsm.time[0]) / np.timedelta64(1, 's')))

# fgsは2.74 secだが、THEMISのspin rateは3 sec

In [ ]:
import xarray as xr
import numpy as np

# low freq.
E8_data_gsm_time    = E8_data_gsm.time
B8_data_gsm         = B16_data_gsm.interp(time=E8_data_gsm_time, method='linear')
vars_8              = ['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B8_gsm_x', 'B8_gsm_y', 'B8_gsm_z']

E8_data_gsm_x, E8_data_gsm_y, E8_data_gsm_z = E8_data_gsm.isel(v_dim=0), E8_data_gsm.isel(v_dim=1), E8_data_gsm.isel(v_dim=2)
B8_data_gsm_x, B8_data_gsm_y, B8_data_gsm_z = B8_data_gsm.isel(v_dim=0), B8_data_gsm.isel(v_dim=1), B8_data_gsm.isel(v_dim=2)

E8_data_gsm_x.name, E8_data_gsm_y.name, E8_data_gsm_z.name, B8_data_gsm_x.name, B8_data_gsm_y.name, B8_data_gsm_z.name  = vars_8

print(E8_data_gsm_x, B8_data_gsm_z)

ds_8_gsm    = xr.merge([E8_data_gsm_x, E8_data_gsm_y, E8_data_gsm_z, B8_data_gsm_x, B8_data_gsm_y, B8_data_gsm_z], compat="minimal")
ds_8_gsm    = ds_8_gsm.dropna(dim='time', how='any', subset=vars_8)

print(ds_8_gsm)

In [ ]:
# high freq.
B128_data_gsm_time  = B128_data_gsm.time
E128_data_gsm       = E512_data_gsm.interp(time=B128_data_gsm_time, method='linear')
vars_128            = ['E128_gsm_x', 'E128_gsm_y', 'E128_gsm_z', 'B128_gsm_x', 'B128_gsm_y', 'B128_gsm_z']

E128_data_gsm_x, E128_data_gsm_y, E128_data_gsm_z = E128_data_gsm.isel(v_dim=0), E128_data_gsm.isel(v_dim=1), E128_data_gsm.isel(v_dim=2)
B128_data_gsm_x, B128_data_gsm_y, B128_data_gsm_z = B128_data_gsm.isel(v_dim=0), B128_data_gsm.isel(v_dim=1), B128_data_gsm.isel(v_dim=2)

E128_data_gsm_x.name, E128_data_gsm_y.name, E128_data_gsm_z.name, B128_data_gsm_x.name, B128_data_gsm_y.name, B128_data_gsm_z.name  = vars_128

print(E128_data_gsm_x, B128_data_gsm_z)

ds_128_gsm    = xr.merge([E128_data_gsm_x, E128_data_gsm_y, E128_data_gsm_z, B128_data_gsm_x, B128_data_gsm_y, B128_data_gsm_z], compat="minimal")
ds_128_gsm    = ds_128_gsm.dropna(dim='time', how='any', subset=vars_128)

print(ds_128_gsm)

In [ ]:
def split_by_gap(ds, time_dim='time', gap_thr=np.timedelta64(30, 's'),
                 prefix='ds_8_gsm_seg'):
    t = ds[time_dim].values
    if t.size == 0:
        return []

    # 先頭はギャップなし。以降で gap_thr を超えたら新セグ開始。
    gaps = np.r_[False, (t[1:] - t[:-1]) > gap_thr]      # shape (Nt,)
    seg_id = np.cumsum(gaps)                              # 0,0,0,1,1,2,...

    ds_tagged = ds.assign_coords(_seg=(time_dim, seg_id))
    segs = [g.drop_vars('_seg') for _, g in ds_tagged.groupby('_seg')]

    return segs

ds_8_gsm_segs = split_by_gap(ds_8_gsm, gap_thr=np.timedelta64(3, 's'))
ds_8_gsm_seg0, ds_8_gsm_seg1, ds_8_gsm_seg2 = ds_8_gsm_segs[:]

print(ds_8_gsm_seg0)
print('')
print(ds_8_gsm_seg1)
print('')
print(ds_8_gsm_seg2)

ds_128_gsm_segs = split_by_gap(ds_128_gsm, gap_thr=np.timedelta64(3, 's'))
ds_128_gsm_seg0, ds_128_gsm_seg1, ds_128_gsm_seg2 = ds_128_gsm_segs[:]

print(ds_128_gsm_seg0)
print('')
print(ds_128_gsm_seg1)
print('')
print(ds_128_gsm_seg2)

In [ ]:
from datetime import datetime
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_8_gsm_seg0_analysis  = ds_8_gsm_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_8_gsm_seg1_analysis  = ds_8_gsm_seg1.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_8_gsm_seg2_analysis  = ds_8_gsm_seg2.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(6, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_8_gsm_seg0_analysis.time, ds_8_gsm_seg0_analysis['E8_gsm_x'], lw=1, c='blue')
#ax_1.plot(ds_8_gsm_seg0_analysis.time, ds_8_gsm_seg0_analysis['E8_gsm_y'], lw=1, c='blue')
#ax_2.plot(ds_8_gsm_seg0_analysis.time, ds_8_gsm_seg0_analysis['E8_gsm_z'], lw=1, c='blue')
#ax_3.plot(ds_8_gsm_seg0_analysis.time, ds_8_gsm_seg0_analysis['B8_gsm_x'], lw=1, c='blue')
#ax_4.plot(ds_8_gsm_seg0_analysis.time, ds_8_gsm_seg0_analysis['B8_gsm_y'], lw=1, c='blue')
#ax_5.plot(ds_8_gsm_seg0_analysis.time, ds_8_gsm_seg0_analysis['B8_gsm_z'], lw=1, c='blue')
#
#ax_0.plot(ds_8_gsm_seg1_analysis.time, ds_8_gsm_seg1_analysis['E8_gsm_x'], lw=1, c='orange')
#ax_1.plot(ds_8_gsm_seg1_analysis.time, ds_8_gsm_seg1_analysis['E8_gsm_y'], lw=1, c='orange')
#ax_2.plot(ds_8_gsm_seg1_analysis.time, ds_8_gsm_seg1_analysis['E8_gsm_z'], lw=1, c='orange')
#ax_3.plot(ds_8_gsm_seg1_analysis.time, ds_8_gsm_seg1_analysis['B8_gsm_x'], lw=1, c='orange')
#ax_4.plot(ds_8_gsm_seg1_analysis.time, ds_8_gsm_seg1_analysis['B8_gsm_y'], lw=1, c='orange')
#ax_5.plot(ds_8_gsm_seg1_analysis.time, ds_8_gsm_seg1_analysis['B8_gsm_z'], lw=1, c='orange')
#
#ax_0.plot(ds_8_gsm_seg2_analysis.time, ds_8_gsm_seg2_analysis['E8_gsm_x'], lw=1, c='green')
#ax_1.plot(ds_8_gsm_seg2_analysis.time, ds_8_gsm_seg2_analysis['E8_gsm_y'], lw=1, c='green')
#ax_2.plot(ds_8_gsm_seg2_analysis.time, ds_8_gsm_seg2_analysis['E8_gsm_z'], lw=1, c='green')
#ax_3.plot(ds_8_gsm_seg2_analysis.time, ds_8_gsm_seg2_analysis['B8_gsm_x'], lw=1, c='green')
#ax_4.plot(ds_8_gsm_seg2_analysis.time, ds_8_gsm_seg2_analysis['B8_gsm_y'], lw=1, c='green')
#ax_5.plot(ds_8_gsm_seg2_analysis.time, ds_8_gsm_seg2_analysis['B8_gsm_z'], lw=1, c='green')
#
#ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'EB_fields_gsm_low_freq_KAW.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_128_gsm_seg0_analysis  = ds_128_gsm_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_128_gsm_seg1_analysis  = ds_128_gsm_seg1.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_128_gsm_seg2_analysis  = ds_128_gsm_seg2.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(6, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_128_gsm_seg0_analysis.time, ds_128_gsm_seg0_analysis['E128_gsm_x'], lw=1, c='blue')
#ax_1.plot(ds_128_gsm_seg0_analysis.time, ds_128_gsm_seg0_analysis['E128_gsm_y'], lw=1, c='blue')
#ax_2.plot(ds_128_gsm_seg0_analysis.time, ds_128_gsm_seg0_analysis['E128_gsm_z'], lw=1, c='blue')
#ax_3.plot(ds_128_gsm_seg0_analysis.time, ds_128_gsm_seg0_analysis['B128_gsm_x'], lw=1, c='blue')
#ax_4.plot(ds_128_gsm_seg0_analysis.time, ds_128_gsm_seg0_analysis['B128_gsm_y'], lw=1, c='blue')
#ax_5.plot(ds_128_gsm_seg0_analysis.time, ds_128_gsm_seg0_analysis['B128_gsm_z'], lw=1, c='blue')
#
#ax_0.plot(ds_128_gsm_seg1_analysis.time, ds_128_gsm_seg1_analysis['E128_gsm_x'], lw=1, c='orange')
#ax_1.plot(ds_128_gsm_seg1_analysis.time, ds_128_gsm_seg1_analysis['E128_gsm_y'], lw=1, c='orange')
#ax_2.plot(ds_128_gsm_seg1_analysis.time, ds_128_gsm_seg1_analysis['E128_gsm_z'], lw=1, c='orange')
#ax_3.plot(ds_128_gsm_seg1_analysis.time, ds_128_gsm_seg1_analysis['B128_gsm_x'], lw=1, c='orange')
#ax_4.plot(ds_128_gsm_seg1_analysis.time, ds_128_gsm_seg1_analysis['B128_gsm_y'], lw=1, c='orange')
#ax_5.plot(ds_128_gsm_seg1_analysis.time, ds_128_gsm_seg1_analysis['B128_gsm_z'], lw=1, c='orange')
#
#ax_0.plot(ds_128_gsm_seg2_analysis.time, ds_128_gsm_seg2_analysis['E128_gsm_x'], lw=1, c='green')
#ax_1.plot(ds_128_gsm_seg2_analysis.time, ds_128_gsm_seg2_analysis['E128_gsm_y'], lw=1, c='green')
#ax_2.plot(ds_128_gsm_seg2_analysis.time, ds_128_gsm_seg2_analysis['E128_gsm_z'], lw=1, c='green')
#ax_3.plot(ds_128_gsm_seg2_analysis.time, ds_128_gsm_seg2_analysis['B128_gsm_x'], lw=1, c='green')
#ax_4.plot(ds_128_gsm_seg2_analysis.time, ds_128_gsm_seg2_analysis['B128_gsm_y'], lw=1, c='green')
#ax_5.plot(ds_128_gsm_seg2_analysis.time, ds_128_gsm_seg2_analysis['B128_gsm_z'], lw=1, c='green')
#
#ax_0.set_ylabel(r'$E_{x}$ (GSM)' + '\n' + '[mV/m]')
#ax_1.set_ylabel(r'$E_{y}$ (GSM)' + '\n' + '[mV/m]')
#ax_2.set_ylabel(r'$E_{z}$ (GSM)' + '\n' + '[mV/m]')
#ax_3.set_ylabel(r'$B_{x}$ (GSM)' + '\n' + '[nT]')
#ax_4.set_ylabel(r'$B_{y}$ (GSM)' + '\n' + '[nT]')
#ax_5.set_ylabel(r'$B_{z}$ (GSM)' + '\n' + '[nT]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#
#ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'EB_fields_gsm_high_freq_KAW.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# GSM -> FACに変換

In [ ]:
psp.themis.state(probe='a', trange=time_range, no_update=True)

In [ ]:
Bspin_data_dt   = (Bspin_data_gsm.time[1] - Bspin_data_gsm.time[0]) / np.timedelta64(1, 's')
B100sec_data_gsm = Bspin_data_gsm.rolling(time=int(100/Bspin_data_dt), center=True).mean()
P60sec_data_gsm     = pt.data_quants['tha_pos_gsm'].interp(time=B100sec_data_gsm.time, method='linear')
P100sec_data_gsm    = P60sec_data_gsm.rolling(time=int(100/Bspin_data_dt), center=True).mean()

B100sec_data_gsm = B100sec_data_gsm.dropna(dim='time', how='any')
pt.store_data('B100sec_data_gsm', data={'x': B100sec_data_gsm.time, 'y': B100sec_data_gsm.data}, attr_dict=B100sec_data_gsm.attrs)
P100sec_data_gsm    = P100sec_data_gsm.dropna(dim='time', how='any')
pt.store_data('P100sec_data_gsm', data={'x': P100sec_data_gsm.time, 'y': P100sec_data_gsm.data}, attr_dict=P60sec_data_gsm.attrs)

psp.fac_matrix_make(mag_var_name='B100sec_data_gsm', other_dim='ygsm', pos_var_name='P100sec_data_gsm', newname='B100_FAC_matrix')

FAC_matrix = pt.data_quants['B100_FAC_matrix'].dropna(dim='time', how='any')

print(FAC_matrix)

In [ ]:
import numpy as np
import xarray as xr

# ---- 入力: FAC_matrix (xarray.DataArray, shape=(time, 3, 3)) ----
fac = FAC_matrix.values

# ---- 検証 ----
# 1. 直交性 (R R^T ≈ I)
orth_err = np.empty(fac.shape[0])
for i in range(fac.shape[0]):
    R = fac[i]
    I = np.eye(3)
    diff = R @ R.T - I
    orth_err[i] = np.linalg.norm(diff)  # Frobeniusノルム

# 2. 行列式
detR = np.linalg.det(fac)

# ---- 結果表示 ----
print("=== FAC_matrix orthogonality check ===")
print(f"orth_err mean: {orth_err.mean():.3e}, max: {orth_err.max():.3e}")
print(f"det(R) mean: {detR.mean():.6f}, std: {detR.std():.3e}")
print(f"det(R) range: {detR.min():.6f} – {detR.max():.6f}")

# しきい値を超えるサンプル検出（例: ノルム誤差 > 1e-6）
bad_idx = np.where((orth_err > 1e-6) | (np.abs(detR) < 0.999) | (np.abs(detR) > 1.001))[0]
if len(bad_idx) == 0:
    print("All matrices are orthonormal within tolerance.")
else:
    print(f"{len(bad_idx)} matrices deviate from orthonormality.")
    print("Example indices:", bad_idx[:10])

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

import matplotlib.pyplot as plt
import matplotlib as mpl
from datetime import datetime

mpl.rcParams['font.size'] = 15

fig = plt.figure(figsize=(10, 5))
ax = fig.add_subplot(111)
ax.plot(FAC_matrix.time, np.rad2deg(np.arccos(FAC_matrix.data[:, 2, 2])), c='blue', lw=1)
ax.minorticks_on()
ax.grid(which='both', alpha=0.5)
ax.set_ylabel(r'∠($\mathbf{B}_{0}$, $\mathbf{e}_{z}$ (GSM))')

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax.set_ylim(4, 23)

fig.tight_layout()

if os.path.isdir(path_base_save_plot):
    fig_path = os.path.join(path_base_save_plot, 'rotation_angle.png')
    print(fig_path)
    fig.savefig(fig_path)
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr

def rotate_gsm_to_fac(ds, fac_mat_da, e_base='E8_gsm', b_base='B8_gsm',
                      out_suffix='_fac', interp_method='linear'):
    """
    ds: xarray.Dataset（例: E8_gsm_x,y,z と B8_gsm_x,y,z を持つ）
    fac_mat_da: (time, 3, 3) の回転行列 DataArray（GSM→FAC）
    e_base/b_base: 先頭名
    out_suffix: 出力成分名の接尾辞（_fac）
    """

    M = fac_mat_da.interp(time=ds.time, method=interp_method)

    def vec3(base):
        v = xr.concat([ds[f'{base}_x'], ds[f'{base}_y'], ds[f'{base}_z']], dim='c')
        return v.transpose('time', 'c').astype(np.float64)  # (time,3)

    def matvec(M, V):
        out = np.einsum('tij,tj->ti', M.values, V.values)
        return xr.DataArray(out, coords={'time': V['time'], 'c': ['x','y','z']},
                            dims=['time','c'])

    def split_drop(V):
        # c 座標を削除して 1D に
        x = V.sel(c='x').reset_coords('c', drop=True)
        y = V.sel(c='y').reset_coords('c', drop=True)
        z = V.sel(c='z').reset_coords('c', drop=True)
        return x, y, z

    E_fac = matvec(M, vec3(e_base))
    B_fac = matvec(M, vec3(b_base))

    Ex, Ey, Ez = split_drop(E_fac)
    Bx, By, Bz = split_drop(B_fac)

    def _stem(name: str) -> str:
        # 末尾のアンダースコア区切りを1つだけ落とす（'E8_gsm'→'E8'）
        return name.rsplit('_', 1)[0] if '_' in name else name
    
    e_stem = _stem(e_base)
    b_stem = _stem(b_base)
    
    return xr.Dataset(
          {f'{e_stem}{out_suffix}_x': Ex,
           f'{e_stem}{out_suffix}_y': Ey,
           f'{e_stem}{out_suffix}_z': Ez,
           f'{b_stem}{out_suffix}_x': Bx,
           f'{b_stem}{out_suffix}_y': By,
           f'{b_stem}{out_suffix}_z': Bz},
           attrs=ds.attrs
    )

In [ ]:
ds_8_fac_seg0   = rotate_gsm_to_fac(ds=ds_8_gsm_seg0, fac_mat_da=FAC_matrix, e_base='E8_gsm', b_base='B8_gsm')
ds_8_fac_seg1   = rotate_gsm_to_fac(ds=ds_8_gsm_seg1, fac_mat_da=FAC_matrix, e_base='E8_gsm', b_base='B8_gsm')
ds_8_fac_seg2   = rotate_gsm_to_fac(ds=ds_8_gsm_seg2, fac_mat_da=FAC_matrix, e_base='E8_gsm', b_base='B8_gsm')

ds_128_fac_seg0   = rotate_gsm_to_fac(ds=ds_128_gsm_seg0, fac_mat_da=FAC_matrix, e_base='E128_gsm', b_base='B128_gsm')
ds_128_fac_seg1   = rotate_gsm_to_fac(ds=ds_128_gsm_seg1, fac_mat_da=FAC_matrix, e_base='E128_gsm', b_base='B128_gsm')
ds_128_fac_seg2   = rotate_gsm_to_fac(ds=ds_128_gsm_seg2, fac_mat_da=FAC_matrix, e_base='E128_gsm', b_base='B128_gsm')

ds_8_fac_seg0   = ds_8_fac_seg0.dropna(dim='time', how='any')
ds_8_fac_seg1   = ds_8_fac_seg1.dropna(dim='time', how='any')
ds_8_fac_seg2   = ds_8_fac_seg2.dropna(dim='time', how='any')

ds_128_fac_seg0   = ds_128_fac_seg0.dropna(dim='time', how='any')
ds_128_fac_seg1   = ds_128_fac_seg1.dropna(dim='time', how='any')
ds_128_fac_seg2   = ds_128_fac_seg2.dropna(dim='time', how='any')

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_8_fac_seg0_analysis  = ds_8_fac_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_8_fac_seg1_analysis  = ds_8_fac_seg1.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_8_fac_seg2_analysis  = ds_8_fac_seg2.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(6, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_8_fac_seg0_analysis.time, ds_8_fac_seg0_analysis['E8_fac_x'], lw=1, c='blue')
#ax_1.plot(ds_8_fac_seg0_analysis.time, ds_8_fac_seg0_analysis['E8_fac_y'], lw=1, c='blue')
#ax_2.plot(ds_8_fac_seg0_analysis.time, ds_8_fac_seg0_analysis['E8_fac_z'], lw=1, c='blue')
#ax_3.plot(ds_8_fac_seg0_analysis.time, ds_8_fac_seg0_analysis['B8_fac_x'], lw=1, c='blue')
#ax_4.plot(ds_8_fac_seg0_analysis.time, ds_8_fac_seg0_analysis['B8_fac_y'], lw=1, c='blue')
#ax_5.plot(ds_8_fac_seg0_analysis.time, ds_8_fac_seg0_analysis['B8_fac_z'], lw=1, c='blue')
#
#ax_0.plot(ds_8_fac_seg1_analysis.time, ds_8_fac_seg1_analysis['E8_fac_x'], lw=1, c='orange')
#ax_1.plot(ds_8_fac_seg1_analysis.time, ds_8_fac_seg1_analysis['E8_fac_y'], lw=1, c='orange')
#ax_2.plot(ds_8_fac_seg1_analysis.time, ds_8_fac_seg1_analysis['E8_fac_z'], lw=1, c='orange')
#ax_3.plot(ds_8_fac_seg1_analysis.time, ds_8_fac_seg1_analysis['B8_fac_x'], lw=1, c='orange')
#ax_4.plot(ds_8_fac_seg1_analysis.time, ds_8_fac_seg1_analysis['B8_fac_y'], lw=1, c='orange')
#ax_5.plot(ds_8_fac_seg1_analysis.time, ds_8_fac_seg1_analysis['B8_fac_z'], lw=1, c='orange')
#
#ax_0.plot(ds_8_fac_seg2_analysis.time, ds_8_fac_seg2_analysis['E8_fac_x'], lw=1, c='green')
#ax_1.plot(ds_8_fac_seg2_analysis.time, ds_8_fac_seg2_analysis['E8_fac_y'], lw=1, c='green')
#ax_2.plot(ds_8_fac_seg2_analysis.time, ds_8_fac_seg2_analysis['E8_fac_z'], lw=1, c='green')
#ax_3.plot(ds_8_fac_seg2_analysis.time, ds_8_fac_seg2_analysis['B8_fac_x'], lw=1, c='green')
#ax_4.plot(ds_8_fac_seg2_analysis.time, ds_8_fac_seg2_analysis['B8_fac_y'], lw=1, c='green')
#ax_5.plot(ds_8_fac_seg2_analysis.time, ds_8_fac_seg2_analysis['B8_fac_z'], lw=1, c='green')
#
#ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'EB_fields_fac_low_freq_KAW.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_128_fac_seg0_analysis  = ds_128_fac_seg0.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_128_fac_seg1_analysis  = ds_128_fac_seg1.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_128_fac_seg2_analysis  = ds_128_fac_seg2.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(6, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_128_fac_seg0_analysis.time, ds_128_fac_seg0_analysis['E128_fac_x'], lw=1, c='blue')
#ax_1.plot(ds_128_fac_seg0_analysis.time, ds_128_fac_seg0_analysis['E128_fac_y'], lw=1, c='blue')
#ax_2.plot(ds_128_fac_seg0_analysis.time, ds_128_fac_seg0_analysis['E128_fac_z'], lw=1, c='blue')
#ax_3.plot(ds_128_fac_seg0_analysis.time, ds_128_fac_seg0_analysis['B128_fac_x'], lw=1, c='blue')
#ax_4.plot(ds_128_fac_seg0_analysis.time, ds_128_fac_seg0_analysis['B128_fac_y'], lw=1, c='blue')
#ax_5.plot(ds_128_fac_seg0_analysis.time, ds_128_fac_seg0_analysis['B128_fac_z'], lw=1, c='blue')
#
#ax_0.plot(ds_128_fac_seg1_analysis.time, ds_128_fac_seg1_analysis['E128_fac_x'], lw=1, c='orange')
#ax_1.plot(ds_128_fac_seg1_analysis.time, ds_128_fac_seg1_analysis['E128_fac_y'], lw=1, c='orange')
#ax_2.plot(ds_128_fac_seg1_analysis.time, ds_128_fac_seg1_analysis['E128_fac_z'], lw=1, c='orange')
#ax_3.plot(ds_128_fac_seg1_analysis.time, ds_128_fac_seg1_analysis['B128_fac_x'], lw=1, c='orange')
#ax_4.plot(ds_128_fac_seg1_analysis.time, ds_128_fac_seg1_analysis['B128_fac_y'], lw=1, c='orange')
#ax_5.plot(ds_128_fac_seg1_analysis.time, ds_128_fac_seg1_analysis['B128_fac_z'], lw=1, c='orange')
#
#ax_0.plot(ds_128_fac_seg2_analysis.time, ds_128_fac_seg2_analysis['E128_fac_x'], lw=1, c='green')
#ax_1.plot(ds_128_fac_seg2_analysis.time, ds_128_fac_seg2_analysis['E128_fac_y'], lw=1, c='green')
#ax_2.plot(ds_128_fac_seg2_analysis.time, ds_128_fac_seg2_analysis['E128_fac_z'], lw=1, c='green')
#ax_3.plot(ds_128_fac_seg2_analysis.time, ds_128_fac_seg2_analysis['B128_fac_x'], lw=1, c='green')
#ax_4.plot(ds_128_fac_seg2_analysis.time, ds_128_fac_seg2_analysis['B128_fac_y'], lw=1, c='green')
#ax_5.plot(ds_128_fac_seg2_analysis.time, ds_128_fac_seg2_analysis['B128_fac_z'], lw=1, c='green')
#
#ax_0.set_ylabel(r'$E_{x}$ (FAC)' + '\n' + '[mV/m]')
#ax_1.set_ylabel(r'$E_{y}$ (FAC)' + '\n' + '[mV/m]')
#ax_2.set_ylabel(r'$E_{z}$ (FAC)' + '\n' + '[mV/m]')
#ax_3.set_ylabel(r'$B_{x}$ (FAC)' + '\n' + '[nT]')
#ax_4.set_ylabel(r'$B_{y}$ (FAC)' + '\n' + '[nT]')
#ax_5.set_ylabel(r'$B_{z}$ (FAC)' + '\n' + '[nT]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'EB_fields_fac_high_freq_KAW.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# Wavelet analysis

In [ ]:
import os
import sys
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pyspedas as psp
import pytplot as pt
import pywt

sys.path.append("..")
import module_handmade.tdwavelet_themis as tw
importlib.reload(tw)

vars_ = ['E128_fac_x','E128_fac_y','E128_fac_z', 'B128_fac_x','B128_fac_y','B128_fac_z']
ds_128_fac_cwt_seg0 = tw.cwt_from_dataset(ds_128_fac_seg0, dt=1/128, s0=2, dj=1/32, variables=vars_)
ds_128_fac_cwt_seg1 = tw.cwt_from_dataset(ds_128_fac_seg1, dt=1/128, s0=2, dj=1/32, variables=vars_)
ds_128_fac_cwt_seg2 = tw.cwt_from_dataset(ds_128_fac_seg2, dt=1/128, s0=2, dj=1/32, variables=vars_)

vars_ = ['E8_fac_x','E8_fac_y','E8_fac_z', 'B8_fac_x','B8_fac_y','B8_fac_z']
ds_8_fac_cwt_seg0 = tw.cwt_from_dataset(ds_8_fac_seg0, dt=1/8, s0=2, dj=1/32, variables=vars_)
ds_8_fac_cwt_seg1 = tw.cwt_from_dataset(ds_8_fac_seg1, dt=1/8, s0=2, dj=1/32, variables=vars_)
ds_8_fac_cwt_seg2 = tw.cwt_from_dataset(ds_8_fac_seg2, dt=1/8, s0=2, dj=1/32, variables=vars_)

print(ds_128_fac_cwt_seg0)
print(ds_8_fac_cwt_seg0)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# ---- セグメント連結（freq合わせ）----
def concat_cwt_segments(dsets, var):
    das = [ds[var] for ds in dsets if var in ds]
    if not das: return None, None
    freqs = np.unique(np.concatenate([da.freq.values for da in das]))
    das = [da if np.array_equal(da.freq.values, freqs) else da.interp(freq=freqs) for da in das]
    pow_cat = xr.concat(das, dim="time").sortby("time").assign_coords(freq=("freq", freqs))
    coi_name = var.replace("_cwt", "_coi")
    coi_cat = xr.concat([ds[coi_name] for ds in dsets if coi_name in ds], dim="time").sortby("time") \
              if all(coi_name in ds for ds in dsets) else None
    return pow_cat, coi_cat

# ---- 1面描画：外でax/caxを用意する ----
def plot_cwt_on_ax(ax, da_pow, da_coi=None, t0=None, minutes=5,
                   zrange=(1e-6, 1e3), yrange=(1e-2, 4.0),
                   cmap="turbo", label_left="", unit_right=""):
    # 時間切り出し
    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        da = da_pow.sel(time=slice(t0, t1))
        coi = da_coi.sel(time=slice(t0, t1)) if da_coi is not None else None
        #ax.set_xlim(t0, t1)
    else:
        da, coi = da_pow, da_coi
    if da.time.size == 0: return None, None

    T = mdates.date2num(da.time.values)
    F = da.freq.values
    Z = da.values.astype(float)

    # COIマスク（低周波側をNaN）
    if coi is not None:
        C = coi.values[:, None]
        Z = np.where(F[None, :] < C, np.nan, Z)

    # メッシュ
    Tm = np.tile(T, (F.size, 1)).T
    Fm = np.tile(F, (T.size, 1))

    pcm = ax.pcolormesh(Tm, Fm, Z, shading="auto",
                        norm=LogNorm(vmin=zrange[0], vmax=zrange[1]), cmap=cmap)

    ax.minorticks_on()
    ax.set_yscale("log")
    ax.set_ylim(yrange[0], yrange[1])
    ax.set_ylabel(f"{label_left}\n[Hz]")

    if t0 is not None:
        t1 = t0 + np.timedelta64(minutes, "m")
        ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))

    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))   # 1分刻み
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.tick_params(axis="x", rotation=0)


    # 右側カラーバー
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    cb = plt.colorbar(pcm, cax=cax)
    cb.set_label(f"{unit_right}")
    return pcm, cb


In [ ]:
#dsets_128 = [ds_128_fac_cwt_seg0, ds_128_fac_cwt_seg1, ds_128_fac_cwt_seg2]
#targets = [
#    ("E128_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B128_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#joined = {}
#for v, _, _ in targets:
#    da, coi = concat_cwt_segments(dsets_128, v)
#    if da is not None: joined[v] = (da, coi)
#
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(9)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets):
#        if v not in joined: continue
#        da, coi = joined[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(0.3, 64.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_high_freq_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
#dsets_8 = [ds_8_fac_cwt_seg0, ds_8_fac_cwt_seg1, ds_8_fac_cwt_seg2]
#targets = [
#    ("E8_fac_x_cwt", r"$E_{x}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E8_fac_y_cwt", r"$E_{y}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E8_fac_z_cwt", r"$E_{z}$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B8_fac_x_cwt", r"$B_{x}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B8_fac_y_cwt", r"$B_{y}$ (FAC)", "[nT$^2$/Hz]"),
#    ("B8_fac_z_cwt", r"$B_{z}$ (FAC)", "[nT$^2$/Hz]"),
#]
#joined = {}
#for v, _, _ in targets:
#    da, coi = concat_cwt_segments(dsets_8, v)
#    if da is not None: joined[v] = (da, coi)
#
#time_windows = [
#    np.datetime64('2022-09-01T20:45') + np.timedelta64(5, 'm')*n
#    for n in range(32)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v, ylab, unit) in zip(axes, targets):
#        if v not in joined: continue
#        da, coi = joined[v]
#        plot_cwt_on_ax(ax, da, coi, t0=t0, minutes=5,
#                       zrange=(1e-6, 1e3), yrange=(1E-2, 4.0),
#                       cmap="turbo", label_left=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_low_freq_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm

# --- 連結（周波数を合わせて縦結合）
def concat_cwt_segments(dsets, var):
    das = [ds[var] for ds in dsets if var in ds]
    if not das: return None, None
    freqs = np.unique(np.concatenate([da.freq.values for da in das]))
    das = [da if np.array_equal(da.freq.values, freqs) else da.interp(freq=freqs) for da in das]
    pow_cat = xr.concat(das, dim="time").sortby("time").assign_coords(freq=("freq", freqs))
    coi_name = var.replace("_cwt", "_coi")
    coi_cat = xr.concat([ds[coi_name] for ds in dsets if coi_name in ds], dim="time").sortby("time") \
              if all(coi_name in ds for ds in dsets) else None
    return pow_cat, coi_cat

# --- 2帯域を同一axに描画
def plot_cwt_dualband_on_ax(ax,
                            da_hi, coi_hi,   # 高周波 (128Hz系)
                            da_lo, coi_lo,   # 低周波 (8Hz系)
                            t0, minutes=5,
                            f_lo=(1e-2, 4.0),
                            f_hi=(4.0, 64.0),
                            zrange=(1e-6, 1e3),
                            cmap="turbo",
                            ylabel="", unit_right=""):

    t1 = t0 + np.timedelta64(minutes, "m")

    # ---- データ切り出し ----
    dah = da_hi.sel(time=slice(t0, t1)) if da_hi is not None else None
    dal = da_lo.sel(time=slice(t0, t1)) if da_lo is not None else None
    coih = coi_hi.sel(time=slice(t0, t1)) if coi_hi is not None else None
    coil = coi_lo.sel(time=slice(t0, t1)) if coi_lo is not None else None

    if (dah is None or dah.time.size == 0) and (dal is None or dal.time.size == 0):
        return None, None

    def _prep(da, coi, fmin, fmax):
        if da is None or da.time.size == 0:
            return None, None, None
        da2 = da.sel(freq=slice(fmin, fmax))
        T = mdates.date2num(da2.time.values)
        F = da2.freq.values
        Z = da2.values.astype(float)
        if coi is not None:
            C = coi.values[:, None]
            Z = np.where(F[None, :] < C, np.nan, Z)
        Tm = np.tile(T, (F.size, 1)).T
        Fm = np.tile(F, (T.size, 1))
        return Tm, Fm, Z

    pcm_h = pcm_l = None
    # 低周波 (8 Hz 系)
    out_lo = _prep(dal, coil, *f_lo)
    if out_lo[0] is not None:
        Tm_l, Fm_l, Z_l = out_lo
        pcm_l = ax.pcolormesh(Tm_l, Fm_l, Z_l, shading="auto",
                              norm=LogNorm(vmin=zrange[0], vmax=zrange[1]),
                              cmap=cmap)

    # 高周波 (128 Hz 系)
    out_hi = _prep(dah, coih, *f_hi)
    if out_hi[0] is not None:
        Tm_h, Fm_h, Z_h = out_hi
        pcm_h = ax.pcolormesh(Tm_h, Fm_h, Z_h, shading="auto",
                              norm=LogNorm(vmin=zrange[0], vmax=zrange[1]),
                              cmap=cmap)

    # ---- 軸設定 ----
    ax.set_yscale("log")
    ax.set_ylim(f_lo[0], f_hi[1])
    ax.set_ylabel(f"{ylabel}\n[Hz]")
    ax.set_xlim(mdates.date2num(np.array([t0, t1], dtype="datetime64[ns]")))
    ax.xaxis.set_major_locator(mdates.MinuteLocator(interval=1))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    ax.minorticks_on()

    # ---- カラーバー ----
    cax = ax.inset_axes([1.01, 0.05, 0.02, 0.9])
    pcm_main = pcm_h if pcm_h is not None else pcm_l
    cb = plt.colorbar(pcm_main, cax=cax)
    cb.set_label(unit_right)

    return (pcm_h, pcm_l), cb

In [ ]:
#dsets_128 = [ds_128_fac_cwt_seg0, ds_128_fac_cwt_seg1, ds_128_fac_cwt_seg2]
#dsets_8   = [ds_8_fac_cwt_seg0,   ds_8_fac_cwt_seg1,   ds_8_fac_cwt_seg2]
#
#targets = [
#    ("E128_fac_x_cwt","E8_fac_x_cwt",  r"$E_x$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_y_cwt","E8_fac_y_cwt",  r"$E_y$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_z_cwt","E8_fac_z_cwt",  r"$E_z$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B128_fac_x_cwt","B8_fac_x_cwt",  r"$B_x$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_y_cwt","B8_fac_y_cwt",  r"$B_y$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_z_cwt","B8_fac_z_cwt",  r"$B_z$ (FAC)", "[nT$^2$/Hz]"),
#]
#
#joined = {}
#for v_hi, v_lo, _, _ in targets:
#    da_hi, coi_hi = concat_cwt_segments(dsets_128, v_hi)
#    da_lo, coi_lo = concat_cwt_segments(dsets_8,   v_lo)
#    if (da_hi is not None) or (da_lo is not None):
#        joined[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)
#
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(12)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v_hi, v_lo, ylab, unit) in zip(axes, targets):
#        if (v_hi, v_lo) not in joined: continue
#        da_hi, coi_hi, da_lo, coi_lo = joined[(v_hi, v_lo)]
#        plot_cwt_dualband_on_ax(ax, da_hi, coi_hi, da_lo, coi_lo,
#                                t0=t0, minutes=5,
#                                f_lo=(1e-2, 4.0), f_hi=(4.0, 64.0),
#                                zrange=(1e-6, 1e3), cmap="turbo",
#                                ylabel=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

# 各成分のNoise(Median)を抽出

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# 入力: ds_8_fac_cwt_seg0
noise_t0, noise_t1 = np.datetime64('2022-09-01T20:50:00'), np.datetime64('2022-09-01T22:10:00')

# 対象変数（存在チェック付き）
vars_8 = [
    'E8_fac_x_cwt','E8_fac_y_cwt','E8_fac_z_cwt',
    'B8_fac_x_cwt','B8_fac_y_cwt','B8_fac_z_cwt'
]
vars_8 = [v for v in vars_8 if v in ds_8_fac_cwt_seg0.data_vars]

# 時間で切り出し → 周波数ごとに時間方向のnanmedian
noise_da_dict = {}
freq_ref = None
for v in vars_8:
    da = ds_8_fac_cwt_seg0[v].sel(time=slice(noise_t0, noise_t1))
    if da.time.size == 0:
        continue
    med = np.nanmedian(da.values, axis=0)  # (freq,)
    freq = da.coords['freq'].values
    if freq_ref is None:
        freq_ref = freq
    noise_da_dict[v] = xr.DataArray(med, dims=['frequency'], coords={'frequency': freq_ref}, name=v)

noise_ds = xr.Dataset(noise_da_dict)
print(noise_ds)

# プロット
fig, ax = plt.subplots(figsize=(8,6))
for v in noise_ds.data_vars:
    # ラベル例: E_x, B_y など
    prefix = v.split('_')[0]   # E8 or B8
    comp   = v.split('_')[2]   # x/y/z
    label  = f"${prefix[0]}_{comp}$"
    ax.loglog(noise_ds['frequency'], noise_ds[v], label=label)

ax.minorticks_on()
ax.set_xlabel('Frequency [Hz]')
ax.set_ylabel('Median PSD')
ax.set_title('Noise floor (median) 20:50–22:10  (E: (mV/m)$^2$/Hz,  B: nT$^2$/Hz)')
ax.grid(True, which='both', ls=':')
ax.legend(ncol=3)
ax.set_xlim(1e-2, 4)
ax.set_ylim(1e-8, 5e1)
plt.tight_layout()

# 保存 or 表示
if os.path.isdir(path_base_save_plot):
    fn = f"noise_median_bs_{str(noise_t0).replace(':','')}_{str(noise_t1).replace(':','')}.png"
    fig.savefig(os.path.join(path_base_save_plot, fn), dpi=300, bbox_inches='tight')
    plt.close(fig)
else:
    plt.show()
    plt.close(fig)

In [ ]:
import xarray as xr
import numpy as np

def make_cwt_clean_segments_8(dsets_8, noise_ds):
    """
    入力:
      dsets_8 : [ds_8_fac_cwt_seg0, ds_8_fac_cwt_seg1, ds_8_fac_cwt_seg2]
      noise_ds: 各成分のノイズ床（frequency or freq 次元, 1D）
    出力:
      cleaned_8: [ds_8_fac_cwt_clean_seg0, ...]  各dsは
                 {E8_fac_{x,y,z}_cwt_clean, B8_fac_{x,y,z}_cwt_clean,
                  E8_fac_{x,y,z}_coi,      B8_fac_{x,y,z}_coi} をdata_varsに持つ
    """
    # 周波数座標名を統一
    def _noise_for(var):
        if var not in noise_ds:
            return None
        nda = noise_ds[var]
        if "frequency" in nda.dims:
            nda = nda.rename({"frequency": "freq"})
        return nda

    target_vars = [
        "E8_fac_x_cwt","E8_fac_y_cwt","E8_fac_z_cwt",
        "B8_fac_x_cwt","B8_fac_y_cwt","B8_fac_z_cwt",
    ]

    cleaned_list = []
    for ds in dsets_8:
        new_vars = {}
        # 座標をそのまま流用
        coords = {"time": ds.time, "freq": ds.freq}

        for v in target_vars:
            if v not in ds:
                continue
            n_da = _noise_for(v)
            if n_da is None:
                continue

            # ノイズ床を各dsのfreqに合わせる
            n_interp = n_da.interp(freq=ds[v].freq)

            # 減算（broadcast）
            cleaned = ds[v] - n_interp
            cleaned = cleaned.where(cleaned > 0)

            new_vars[f"{v}_clean"] = xr.DataArray(
                cleaned.astype(np.float32),
                dims=("time", "freq"),
                coords=coords,
                attrs={**ds[v].attrs, "noise_removed": True}
            )

            # 対応するCOIをそのまま持たせる
            coi_name = v.replace("_cwt", "_coi")
            if coi_name in ds:
                new_vars[coi_name] = ds[coi_name]

        cleaned_list.append(xr.Dataset(new_vars, coords=coords))

    return cleaned_list

# 使い方
dsets_8 = [ds_8_fac_cwt_seg0, ds_8_fac_cwt_seg1, ds_8_fac_cwt_seg2]
ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2 = make_cwt_clean_segments_8(dsets_8, noise_ds)

print(ds_8_fac_cwt_clean_seg0)

In [ ]:
#dsets_128 = [ds_128_fac_cwt_seg0, ds_128_fac_cwt_seg1, ds_128_fac_cwt_seg2]
#dsets_8_clean   = [ds_8_fac_cwt_clean_seg0,   ds_8_fac_cwt_clean_seg1,   ds_8_fac_cwt_clean_seg2]
#
#targets = [
#    ("E128_fac_x_cwt","E8_fac_x_cwt_clean",  r"$E_x$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_y_cwt","E8_fac_y_cwt_clean",  r"$E_y$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("E128_fac_z_cwt","E8_fac_z_cwt_clean",  r"$E_z$ (FAC)", "[(mV/m)$^2$/Hz]"),
#    ("B128_fac_x_cwt","B8_fac_x_cwt_clean",  r"$B_x$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_y_cwt","B8_fac_y_cwt_clean",  r"$B_y$ (FAC)", "[nT$^2$/Hz]"),
#    ("B128_fac_z_cwt","B8_fac_z_cwt_clean",  r"$B_z$ (FAC)", "[nT$^2$/Hz]"),
#]
#
#joined = {}
#for v_hi, v_lo, _, _ in targets:
#    da_hi, coi_hi = concat_cwt_segments(dsets_128, v_hi)
#    da_lo, coi_lo = concat_cwt_segments(dsets_8_clean,   v_lo)
#    if (da_hi is not None) or (da_lo is not None):
#        joined[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)
#
#time_windows = [
#    np.datetime64('2022-09-01T22:25') + np.timedelta64(5, 'm')*n
#    for n in range(12)
#]
#
#for t0 in time_windows:
#    fig, axes = plt.subplots(len(targets), 1, figsize=(10, 10), sharex=True)
#    for ax, (v_hi, v_lo, ylab, unit) in zip(axes, targets):
#        if (v_hi, v_lo) not in joined: continue
#        da_hi, coi_hi, da_lo, coi_lo = joined[(v_hi, v_lo)]
#        plot_cwt_dualband_on_ax(ax, da_hi, coi_hi, da_lo, coi_lo,
#                                t0=t0, minutes=5,
#                                f_lo=(1e-2, 4.0), f_hi=(4.0, 64.0),
#                                zrange=(1e-6, 1e3), cmap="turbo",
#                                ylabel=ylab, unit_right=unit)
#
#    axes[-1].set_xlabel("time")
#    fig.tight_layout()
#
#    if os.path.isdir(path_base_save_plot):
#        t0_str = str(t0)
#        fn_time = t0_str.replace(':', '').replace('T', '_')
#        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}.png')
#        print(fig_path)
#        fig.savefig(fig_path)
#        plt.close(fig)
#    else:
#        plt.show()
#        plt.close(fig)

In [ ]:
mu_0    = 4.*np.pi*1E-7

Ex_fac  = xr.concat([ds_8_fac_seg0['E8_fac_x'], ds_8_fac_seg1['E8_fac_x'], ds_8_fac_seg2['E8_fac_x']], dim='time')
Ey_fac  = xr.concat([ds_8_fac_seg0['E8_fac_y'], ds_8_fac_seg1['E8_fac_y'], ds_8_fac_seg2['E8_fac_y']], dim='time')
Ez_fac  = xr.concat([ds_8_fac_seg0['E8_fac_z'], ds_8_fac_seg1['E8_fac_z'], ds_8_fac_seg2['E8_fac_z']], dim='time')
Bx_fac  = xr.concat([ds_8_fac_seg0['B8_fac_x'], ds_8_fac_seg1['B8_fac_x'], ds_8_fac_seg2['B8_fac_x']], dim='time')
By_fac  = xr.concat([ds_8_fac_seg0['B8_fac_y'], ds_8_fac_seg1['B8_fac_y'], ds_8_fac_seg2['B8_fac_y']], dim='time')
Bz_fac  = xr.concat([ds_8_fac_seg0['B8_fac_z'], ds_8_fac_seg1['B8_fac_z'], ds_8_fac_seg2['B8_fac_z']], dim='time')

S_para  = (Ex_fac * By_fac - Ey_fac * Bx_fac) / mu_0 * 1E-12

print(S_para)

In [ ]:
dsets_128 = [ds_128_fac_cwt_seg0, ds_128_fac_cwt_seg1, ds_128_fac_cwt_seg2]
dsets_8_clean   = [ds_8_fac_cwt_clean_seg0,   ds_8_fac_cwt_clean_seg1,   ds_8_fac_cwt_clean_seg2]

targets = [
    ("E128_fac_x_cwt","E8_fac_x_cwt_clean",  r"$E_x$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_y_cwt","E8_fac_y_cwt_clean",  r"$E_y$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("E128_fac_z_cwt","E8_fac_z_cwt_clean",  r"$E_z$ (FAC)", "[(mV/m)$^2$/Hz]"),
    ("B128_fac_x_cwt","B8_fac_x_cwt_clean",  r"$B_x$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_y_cwt","B8_fac_y_cwt_clean",  r"$B_y$ (FAC)", "[nT$^2$/Hz]"),
    ("B128_fac_z_cwt","B8_fac_z_cwt_clean",  r"$B_z$ (FAC)", "[nT$^2$/Hz]"),
]

joined = {}
for v_hi, v_lo, _, _ in targets:
    da_hi, coi_hi = concat_cwt_segments(dsets_128, v_hi)
    da_lo, coi_lo = concat_cwt_segments(dsets_8_clean,   v_lo)
    if (da_hi is not None) or (da_lo is not None):
        joined[(v_hi, v_lo)] = (da_hi, coi_hi, da_lo, coi_lo)

time_windows = [
    np.datetime64('2022-09-01T22:30:00'),
    np.datetime64('2022-09-01T22:47:30'),
    np.datetime64('2022-09-01T23:05:00')
]

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

for t0 in time_windows:
    fig, axes = plt.subplots(len(targets)+1, 1, figsize=(10, 12), sharex=True)
    axes_cwt = axes[:len(targets)]
    for ax, (v_hi, v_lo, ylab, unit) in zip(axes_cwt, targets):
        if (v_hi, v_lo) not in joined: continue
        da_hi, coi_hi, da_lo, coi_lo = joined[(v_hi, v_lo)]
        plot_cwt_dualband_on_ax(ax, da_hi, coi_hi, da_lo, coi_lo,
                                t0=t0, minutes=5,
                                f_lo=(1e-2, 4.0), f_hi=(4.0, 64.0),
                                zrange=(1e-6, 1e3), cmap="turbo",
                                ylabel=ylab, unit_right=unit)
    
    S_para_window   = S_para.sel(time=slice(t0, t0+np.timedelta64(5, 'm')))
    axes[len(targets)].plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    axes[len(targets)].set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    axes[len(targets)].minorticks_on()
    axes[len(targets)].grid(which='both', alpha=0.5)
    axes[-1].set_xlabel("time")
    fig.tight_layout()
    fig.subplots_adjust(hspace=0.1)

    add_panel_label(axes[0], '(1)')
    add_panel_label(axes[1], '(2)')
    add_panel_label(axes[2], '(3)')
    add_panel_label(axes[3], '(4)')
    add_panel_label(axes[4], '(5)')
    add_panel_label(axes[5], '(6)')
    add_panel_label(axes[6], '(7)')

    if os.path.isdir(path_base_save_plot):
        t0_str = str(t0)
        fn_time = t0_str.replace(':', '').replace('T', '_')
        fig_path = os.path.join(path_base_save_plot, f'EB_fields_fac_cwt_clean_{fn_time}_for_figure.png')
        print(fig_path)
        fig.savefig(fig_path)
        plt.close(fig)
    else:
        plt.show()
        plt.close(fig)

# 軌道データから、衛星速度を導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.themis.state(
    probe='a',
    trange=time_range
)

vel_gsm     = pt.data_quants['tha_vel_gsm']

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#vel_gsm_analysis  = vel_gsm.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(vel_gsm_analysis.time, vel_gsm_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(vel_gsm_analysis.time, vel_gsm_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(vel_gsm_analysis.time, vel_gsm_analysis.data[:, 2], lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (GSM)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (GSM)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (GSM)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_gsm.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
import numpy as np
import xarray as xr

def rotate_vec3_gsm_to_fac(da_vec, fac_mat_da, out_suffix="_fac", interp_method="linear"):
    """
    GSM→FAC回転を v_dim=3 のベクトル DataArray に適用する。

    Parameters
    ----------
    da_vec : xarray.DataArray
        形状 (time, v_dim=3) のベクトルデータ。座標 'time' と 'v_dim' を含む。
    fac_mat_da : xarray.DataArray
        形状 (time, 3, 3) の回転行列（GSM→FAC変換行列）。
    out_suffix : str
        出力名の接尾辞。
    interp_method : str
        時間補間法。'linear' など。

    Returns
    -------
    da_fac : xarray.DataArray
        形状 (time, v_dim=3) の回転後ベクトル。
        名前は da_vec.name + out_suffix。
    """
    # 時間軸を補間
    M = fac_mat_da.interp(time=da_vec.time, method=interp_method)

    # einsum でベクトル回転
    V_in = da_vec.transpose("time", "v_dim").astype(np.float64)
    V_out = np.einsum("tij,tj->ti", M.values, V_in.values)  # (time, 3)

    # 結果をDataArrayに再構成
    da_fac = xr.DataArray(
        V_out.astype(np.float32),
        dims=("time", "v_dim"),
        coords={"time": da_vec.time, "v_dim": da_vec.v_dim},
        name=(f"{da_vec.name}{out_suffix}" if da_vec.name else None),
        attrs={**da_vec.attrs, "rotated": "GSM→FAC"},
    )
    return da_fac



In [ ]:
vel_fac = rotate_vec3_gsm_to_fac(vel_gsm, FAC_matrix)
vel_fac = vel_fac.dropna(dim='time', how='any')
print(vel_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#vel_fac_analysis  = vel_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(vel_fac_analysis.time, vel_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(vel_fac_analysis.time, vel_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(vel_fac_analysis.time, vel_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(vel_fac_analysis.time, np.sqrt(vel_fac_analysis.data[:, 0]**2E0 + vel_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sc}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sc}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sc}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sc}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sc_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# イオン流速($\approx$ MHD流速)、イオン温度$\rightarrow$イオン熱速度、電子温度$\rightarrow$ ion acoustic sppedの導出

In [ ]:
import pyspedas as psp
import pytplot as pt

psp.themis.mom(trange=time_range, probe='a', level='l2', no_update=True)

In [ ]:
ND_electron     = pt.data_quants['tha_peem_density']        # [/cc]
Temp_electron   = pt.data_quants['tha_peem_ptot']           # [eV]

ND_ion          = pt.data_quants['tha_peim_density']        # [/cc]
Temp_ion        = pt.data_quants['tha_peim_ptot']           # [eV]
Velocity_ion_gsm= pt.data_quants['tha_peim_velocity_gsm']   # [km/s]

In [ ]:
Velocity_ion_fac    = rotate_vec3_gsm_to_fac(Velocity_ion_gsm, FAC_matrix)
Velocity_ion_fac    = Velocity_ion_fac.dropna(dim='time', how='any')
print(Velocity_ion_fac)
print((Velocity_ion_fac.time[1] - Velocity_ion_fac.time[0]) / np.timedelta64(1, 's'))

In [ ]:
Velocity_sys_fac = Velocity_ion_fac - vel_fac.interp(time=Velocity_ion_fac.time, method='linear')
Velocity_sys_fac = Velocity_sys_fac.dropna(dim='time', how='any')
print(Velocity_sys_fac)

In [ ]:
#import matplotlib.pyplot as plt
#
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#Velocity_ion_gsm_analysis  = Velocity_ion_gsm.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(3, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(Velocity_ion_gsm_analysis.time, Velocity_ion_gsm_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(Velocity_ion_gsm_analysis.time, Velocity_ion_gsm_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(Velocity_ion_gsm_analysis.time, Velocity_ion_gsm_analysis.data[:, 2], lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (GSM)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (GSM)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (GSM)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#
#ax_2.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_gsm.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#Velocity_ion_fac_analysis  = Velocity_ion_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(Velocity_ion_fac_analysis.time, Velocity_ion_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(Velocity_ion_fac_analysis.time, Velocity_ion_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(Velocity_ion_fac_analysis.time, Velocity_ion_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(Velocity_ion_fac_analysis.time, np.sqrt(Velocity_ion_fac_analysis.data[:, 0]**2E0 + Velocity_ion_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{ion}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{ion}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{ion}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{ion}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_ion_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#Velocity_sys_fac_analysis  = Velocity_sys_fac.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(Velocity_sys_fac_analysis.time, Velocity_sys_fac_analysis.data[:, 0], lw=1, c='k')
#ax_1.plot(Velocity_sys_fac_analysis.time, Velocity_sys_fac_analysis.data[:, 1], lw=1, c='k')
#ax_2.plot(Velocity_sys_fac_analysis.time, Velocity_sys_fac_analysis.data[:, 2], lw=1, c='k')
#ax_3.plot(Velocity_sys_fac_analysis.time, np.sqrt(Velocity_sys_fac_analysis.data[:, 0]**2E0 + Velocity_sys_fac_analysis.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#dt_Velocity_sys_fac = (Velocity_sys_fac.time.data[1] - Velocity_sys_fac.time.data[0]) / np.timedelta64(1, 's')
#Velocity_sys_fac_mean = Velocity_sys_fac.rolling(time=int(100/dt_Velocity_sys_fac), center=True).mean()
#Velocity_sys_fac_analysis_mean  = Velocity_sys_fac_mean.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(4, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(Velocity_sys_fac_analysis_mean.time, Velocity_sys_fac_analysis_mean.data[:, 0], lw=1, c='k')
#ax_1.plot(Velocity_sys_fac_analysis_mean.time, Velocity_sys_fac_analysis_mean.data[:, 1], lw=1, c='k')
#ax_2.plot(Velocity_sys_fac_analysis_mean.time, Velocity_sys_fac_analysis_mean.data[:, 2], lw=1, c='k')
#ax_3.plot(Velocity_sys_fac_analysis_mean.time, np.sqrt(Velocity_sys_fac_analysis_mean.data[:, 0]**2E0 + Velocity_sys_fac_analysis_mean.data[:, 1]**2E0), lw=1, c='k')
#
#ax_0.set_ylabel(r'$V_{\mathrm{sys}x}$ (FAC)' + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$V_{\mathrm{sys}y}$ (FAC)' + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$V_{\mathrm{sys}z}$ (FAC)' + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$V_{\mathrm{sys}\perp}$ (FAC)' + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#
#ax_3.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'v_sys_fac_interp.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

- Alfvén speed
```math
v_{\mathrm{A}} := \frac{B_{0}}{\sqrt{\mu_{0} n_{\mathrm{e}} m_{\mathrm{p}}}}
```
- Ion (proton) thermal speed
```math
v_{\mathrm{thp}} := \sqrt{\frac{2 T_{\mathrm{i}}}{m_{\mathrm{p}}}}
```
- Ion (proton) acoustic speed
```math
c_{\mathrm{s}} := \sqrt{\frac{T_{\mathrm{e}}}{m_{\mathrm{p}}}}
```
- Proton cyclotron frequency
```math
f_{\mathrm{p}} := \frac{1}{2 \pi} \frac{e B_{0}}{m_{\mathrm{p}}}
```
- Ion plasma beta
```math
\beta_{\mathrm{i}} := \frac{2 \mu_{0} n_{\mathrm{e}} T_{\mathrm{i}}}{B_{0}^{2}} = \left( \frac{v_{\mathrm{thp}}}{v_{\mathrm{A}}} \right)^{2}
```
- Ion-to-electron temperature ratio
```math
\tau := \frac{T_{\mathrm{i}}}{T_{\mathrm{e}}} = \frac{1}{2} \left( \frac{v_{\mathrm{thp}}}{c_{\mathrm{s}}} \right)^{2}
```

計算後、ds_velocityにAlfvén speed, ion thermal speed, ion acoustic speed, perpendicular system speedを格納。
ds_parameterにion plasma beta, ion-to-electron temperature ratio, proton cyclotron frequencyを格納。

In [ ]:
B_total = pt.data_quants['tha_fgs_btotal']  # nT

time_base = B_total.time
print(time_base)

ND_electron_interp      = ND_electron.interp(time=time_base, method='linear')
Temp_electron_interp    = Temp_electron.interp(time=time_base, method='linear')
Temp_ion_interp         = Temp_ion.interp(time=time_base, method='linear')
Velocity_sys_fac_interp = Velocity_sys_fac.interp(time=time_base, method='linear')
Velocity_sys_fac_perp_interp    = xr.DataArray(
    data=np.sqrt(Velocity_sys_fac_interp.data[:, 0]**2E0 + Velocity_sys_fac_interp.data[:, 1]**2E0)*1E3,
    dims=['time'],
    coords={'time': Velocity_sys_fac_interp.time},
    attrs=Velocity_sys_fac_interp.attrs
)

Velocity_sys_fac_perp_interp.attrs['UNITS'] = 'm/s (ALL Qs)'
print(Velocity_sys_fac_perp_interp.attrs)

proton_mass = 1.6726219e-27  # kg
elementary_charge = 1.60218e-19  # C
mu0 = 4*np.pi*1e-7

Alfven_speed        = B_total*1E-9 / np.sqrt(mu0 * ND_electron_interp*1E6 * proton_mass)
ion_thermal_speed   = np.sqrt(2E0 * Temp_ion_interp*elementary_charge / proton_mass)
ion_acoustic_speed  = np.sqrt(Temp_electron_interp*elementary_charge / proton_mass)

electron_mass_kg        = 9.1093837E-31
electron_thermal_speed  = np.sqrt(2E0 * Temp_electron_interp*elementary_charge / electron_mass_kg)

proton_cycl_freq    = elementary_charge * B_total*1E-9 / proton_mass / 2E0 / np.pi

ion_plasma_beta     = (ion_thermal_speed / Alfven_speed)**2E0
ion_to_electron_temp_ratio  = (ion_thermal_speed / ion_acoustic_speed)**2E0 / 2E0

# moving mean
dt_time_base            = (time_base.data[1] - time_base.data[0]) / np.timedelta64(1, 's')

Alfven_speed_mean       = Alfven_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_thermal_speed_mean  = ion_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
electron_thermal_speed_mean = electron_thermal_speed.rolling(time=int(100/dt_time_base), center=True).mean()
ion_acoustic_speed_mean = ion_acoustic_speed.rolling(time=int(100/dt_time_base), center=True).mean()
Velocity_sys_perp_mean  = Velocity_sys_fac_perp_interp.rolling(time=int(100/dt_time_base), center=True).mean()

ion_plasma_beta_mean            = ion_plasma_beta.rolling(time=int(100/dt_time_base), center=True).mean()
ion_to_electron_temp_ratio_mean = ion_to_electron_temp_ratio.rolling(time=int(100/dt_time_base), center=True).mean()
proton_cycl_freq_mean           = proton_cycl_freq.rolling(time=int(100/dt_time_base), center=True).mean()


# DataSet格納
ds_velocity_ms = xr.Dataset(
    {
        'Alfven_speed':         Alfven_speed_mean,
        'ion_thermal_speed':    ion_thermal_speed_mean,
        'electron_thermal_speed':   electron_thermal_speed_mean,
        'ion_acoustic_speed':   ion_acoustic_speed_mean,
        'perp_sys_speed':       Velocity_sys_perp_mean
    }
)
ds_velocity_ms = ds_velocity_ms.dropna(dim='time', how='any')

print(ds_velocity_ms)

ds_parameter = xr.Dataset(
    {
        'ion_plasma_beta':      ion_plasma_beta_mean,
        'i-e_temp_ratio':       ion_to_electron_temp_ratio_mean,
        'proton_cycl_freq_Hz':  proton_cycl_freq_mean,
        'number_density_cc':    ND_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_ion_eV':          Temp_ion_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'temp_electron_eV':     Temp_electron_interp.rolling(time=int(100/dt_time_base), center=True).mean(),
        'B_total_nT':           B_total.rolling(time=int(100/dt_time_base), center=True).mean()
    }
)
ds_parameter = ds_parameter.dropna(dim='time', how='any')

print(ds_parameter)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_velocity_ms_analysis = ds_velocity_ms.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 10))
#gs = fig.add_gridspec(5, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k')
#ax_1.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=1, c='k')
#ax_2.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3,  lw=1, c='k')
#ax_3.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='k')
#ax_4.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$v_{\mathrm{A}}$'         + '\n' + '[km/s]')
#ax_1.set_ylabel(r'$v_{\mathrm{thi}}$'       + '\n' + '[km/s]')
#ax_2.set_ylabel(r'$v_{\mathrm{the}}$'       + '\n' + '[km/s]')
#ax_3.set_ylabel(r'$c_{\mathrm{s}}$'         + '\n' + '[km/s]')
#ax_4.set_ylabel(r'$V_{\mathrm{sys}\perp}$'  + '\n' + '[km/s]')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_4.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'velocity_summary.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#
#mpl.rcParams['font.size'] = 15
#
#fig = plt.figure(figsize=(10, 15))
#gs = fig.add_gridspec(7, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_cc'],   lw=1, c='k')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],         lw=1, c='k')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],    lw=1, c='k')
#ax_3.plot(ds_parameter_analysis.time, ds_parameter_analysis['i-e_temp_ratio'],      lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta'],     lw=1, c='k')
#ax_5.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],          lw=1, c='k')
#ax_6.plot(ds_parameter_analysis.time, ds_parameter_analysis['proton_cycl_freq_Hz'], lw=1, c='k')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'     + '\n' + '[/cc]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$'     + '\n' + '[eV]')
#ax_2.set_ylabel(r'$T_{\mathrm{e}}$'     + '\n' + '[eV]')
#ax_3.set_ylabel(r'$\tau$')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$B_{0}$'              + '\n' + '[nT]')
#ax_6.set_ylabel(r'$f_{\mathrm{H}^{+}}$' + '\n' + '[Hz]')
#
#ax_3.set_yscale('log')
#ax_4.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_6.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#fig.tight_layout()
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'parameter_summary.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

In [ ]:
psp.themis.state(trange=time_range, probe='a', no_update=True)
psp.cotrans(name_in='tha_pos_gsm', name_out='tha_pos_sm', coord_in='gsm', coord_out='sm')
THA_SM_pos = pt.data_quants['tha_pos_sm'].sel(time=slice(time_range_T[0], time_range_T[1]))
print(THA_SM_pos)

THA_rmlatmlt_R = np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0 + THA_SM_pos.data[:, 2]**2E0) / 6378.1
THA_rmlatmlt_MLAT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 2], np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0)))
THA_rmlatmlt_MLT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 1], THA_SM_pos.data[:, 0])) / 15. + 12.

print(THA_rmlatmlt_R)
print(THA_rmlatmlt_MLAT)
print(THA_rmlatmlt_MLT)

```math
\theta_{\mathrm{GSM}} := \mathrm{arctan} \left( \frac{B_{\mathrm{GSM}z}}{\sqrt{B_{\mathrm{GSM}x}^{2} + B_{\mathrm{GSM}y}^{2}}} \right)
```
[Lui et al., 1999; Duan et al., 2011]

In [ ]:
theta_GSM = np.rad2deg(np.arctan(B16_data_gsm.data[:, 2] / np.sqrt(B16_data_gsm.data[:, 0]**2E0 + B16_data_gsm.data[:, 1]**2E0)))

da_theta_GSM = xr.DataArray(
    data=theta_GSM,
    dims=['time'],
    coords={'time': B16_data_gsm.time},
    name='theta_GSM_deg'
)

da_theta_dt  = (da_theta_GSM.time[1] - da_theta_GSM.time[0]) / np.timedelta64(1, 's')
da_theta_GSM = da_theta_GSM.rolling(time=int(100/da_theta_dt), center=True).mean()

da_theta_GSM

In [ ]:
da_B16_data_gsm_z = xr.DataArray(
    data=B16_data_gsm.data[:, 2],
    dims=['time'],
    coords={'time': B16_data_gsm.time},
    name='B16_gsm_z'
)

da_B16_data_gsm_z = da_B16_data_gsm_z.rolling(time=int(100/da_theta_dt), center=True).mean()

time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window = da_B16_data_gsm_z.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

mpl.rcParams['font.size'] = 25
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(da_T_window.time, da_T_window.data, c='k', lw=1)
ax.minorticks_on()
ax.set_ylabel(r'$B_{\mathrm{GSM}z}$' + '\n[nT]')
ax.grid(which='both', alpha=0.5)
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

fig.tight_layout()
plt.show(fig)

In [ ]:
#time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
#time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
#
#ds_parameter_analysis   = ds_parameter.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#ds_velocity_ms_analysis = ds_velocity_ms.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#da_theta_GSM_analysis   = da_theta_GSM.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
#
#import matplotlib.pyplot as plt
#import matplotlib as mpl
#import matplotlib.ticker as mticker
#from datetime import datetime
#
#mpl.rcParams['font.size'] = 25
#
#fig = plt.figure(figsize=(11, 21))
#gs = fig.add_gridspec(8, 1)
#ax_0 = fig.add_subplot(gs[0, 0])
#ax_1 = fig.add_subplot(gs[1, 0], sharex=ax_0)
#ax_2 = fig.add_subplot(gs[2, 0], sharex=ax_0)
#ax_2_share = ax_2.twinx()
#ax_3 = fig.add_subplot(gs[3, 0], sharex=ax_0)
#ax_4 = fig.add_subplot(gs[4, 0], sharex=ax_0)
#ax_5 = fig.add_subplot(gs[5, 0], sharex=ax_0)
#ax_6 = fig.add_subplot(gs[6, 0], sharex=ax_0)
#ax_7 = fig.add_subplot(gs[7, 0], sharex=ax_0)
#
#ax_0.tick_params(axis='x', which='both', labelbottom=False)
#ax_1.tick_params(axis='x', which='both', labelbottom=False)
#ax_2.tick_params(axis='x', which='both', labelbottom=False)
#ax_3.tick_params(axis='x', which='both', labelbottom=False)
#ax_4.tick_params(axis='x', which='both', labelbottom=False)
#ax_5.tick_params(axis='x', which='both', labelbottom=False)
#ax_6.tick_params(axis='x', which='both', labelbottom=False)
#
#ax_0.plot(ds_parameter_analysis.time, ds_parameter_analysis['number_density_cc'],           lw=1, c='k')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_ion_eV'],                 lw=1, c='red', label=r'$T_{\mathrm{i}}$')
#ax_1.plot(ds_parameter_analysis.time, ds_parameter_analysis['temp_electron_eV'],            lw=1, c='blue', label=r'$T_{\mathrm{e}}$')
#ax_2.plot(ds_parameter_analysis.time, ds_parameter_analysis['proton_cycl_freq_Hz'],         lw=1, c='k')
#ax_2_share.plot(ds_parameter_analysis.time, ds_parameter_analysis['B_total_nT'],            lw=1, c='k')
#ax_3.plot(da_theta_GSM_analysis.time, da_theta_GSM_analysis.data,                           lw=1, c='k')
#ax_4.plot(ds_parameter_analysis.time, ds_parameter_analysis['ion_plasma_beta'],             lw=1, c='k')
#ax_5.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['electron_thermal_speed']*1E-3, lw=2, c='b', linestyle='-.')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['Alfven_speed']*1E-3,       lw=1, c='k', label=r'$v_{\mathrm{A}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_thermal_speed']*1E-3,  lw=2, c='red', linestyle='-.', label=r'$v_{\mathrm{thi}}$')
#ax_6.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['ion_acoustic_speed']*1E-3, lw=1, c='magenta', label=r'$c_{\mathrm{s}}$')
#ax_7.plot(ds_velocity_ms_analysis.time, ds_velocity_ms_analysis['perp_sys_speed']*1E-3,     lw=1, c='k')
#
#ax_0.set_ylabel(r'$n_{\mathrm{e}}$'                     + '\n' + '[/cc]')
#ax_1.set_ylabel(r'$T_{\mathrm{i}}$, $T_{\mathrm{e}}$'   + '\n' + '[eV]')
#ax_2.set_ylabel(r'$f_{\mathrm{H}^{+}}$'                 + '\n' + '[Hz]')
#ax_2_share.set_ylabel(r'$B_{0}$'                        + '\n' + '[nT]')
#ax_3.set_ylabel(r'$\theta_{\mathrm{GSM}}$'              + '\n' + r'[deg]')
#ax_4.set_ylabel(r'$\beta_{\mathrm{i}}$')
#ax_5.set_ylabel(r'$v_{\mathrm{the}}$'                   + '\n' + '[km/s]')
#ax_6.set_ylabel(r'$v_{\mathrm{A}}$, $v_{\mathrm{thi}}$, $c_{\mathrm{s}}$'   + '\n' + '[km/s]')
#ax_7.set_ylabel(r'$V_{\mathrm{sys}\perp}$'              + '\n' + '[km/s]')
#
##ax_0.set_yscale('log')
##ax_1.set_yscale('log')
#ax_1.set_ylim(ymin=1)
#ax_4.set_yscale('log')
##ax_5.set_yscale('log')
#
#ax_0.minorticks_on()
#ax_0.grid(which='both', alpha=0.5)
#ax_1.minorticks_on()
#ax_1.grid(which='both', alpha=0.5)
#ax_2.minorticks_on()
#ax_2.grid(which='both', alpha=0.5)
#ax_3.minorticks_on()
#ax_3.grid(which='both', alpha=0.5)
#ax_4.minorticks_on()
#ax_4.grid(which='both', alpha=0.5)
#ax_5.minorticks_on()
#ax_5.grid(which='both', alpha=0.5)
#ax_6.minorticks_on()
#ax_6.grid(which='both', alpha=0.5)
#ax_7.minorticks_on()
#ax_7.grid(which='both', alpha=0.5)
#
#ax_1.legend(fontsize=13, ncol=2)
#ax_6.legend(fontsize=13, ncol=3)
#
#to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
#time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
#ax_7.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
#
#def add_panel_label(ax, label, x=-0.15, y=0.95):
#    ax.text(x, y, label, transform=ax.transAxes,
#            ha='right', va='bottom', clip_on=False)
#
#def to_py_datetime(t_np64):
#    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)
#
## 軌道データ
#t_pos_py = to_py_datetime(THA_SM_pos.time.values)
#t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数
#
#R   = np.asarray(THA_rmlatmlt_R, dtype=float)  # Re
#mlat= np.asarray(THA_rmlatmlt_MLAT, dtype=float)  # deg
#mlt = np.asarray(THA_rmlatmlt_MLT, dtype=float)  # hour [0,24)
#
## --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
#mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)
#
## 補間関数（tick の x は「日数」なのでそのまま使う）
#def interp_at(x_num):
#    Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
#    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
#    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
#    mlti  = np.mod(mltiu, 24.0)
#    return Ri, mlati, mlti
#
## 目盛フォーマッタ
#def rmlt_formatter(x, pos=None):
#    Ri, mlati, mlti = interp_at(x)
#    if np.any(~np.isfinite([Ri, mlati, mlti])):
#        return ""  # 範囲外は空
#    return (f"{Ri:0.2f}\n"
#            f"{mlati:0.2f}\n"
#            f"{mlti:0.2f}")
#
## セカンダリ x 軸（底 side）を作ってラベルを差し替え
#secax = ax_7.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
#secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))
#
## メインの時間ラベルと重ならないよう余白を広げる
#ax_7.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
#secax.tick_params(axis='x', which='major', pad=60)  # R/MLAT/MLTラベル
#
## 好みで：目盛間隔をメイン x と合わせる
#secax.set_ticks(ax_7.get_xticks())
#
#fig.text(0.05, 0.085, "hhmm", ha='center', va='center')
#fig.text(0.05, 0.048, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
#fig.text(0.05, 0.028, r"MLAT", ha='center', va='center')
#fig.text(0.05, 0.008, r"MLT", ha='center', va='center')
#
#add_panel_label(ax_0, '(b-1)')
#add_panel_label(ax_1, '(b-2)')
#add_panel_label(ax_2, '(b-3)')
#add_panel_label(ax_3, '(b-4)')
#add_panel_label(ax_4, '(b-5)')
#add_panel_label(ax_5, '(b-6)')
#add_panel_label(ax_6, '(b-7)')
#add_panel_label(ax_7, '(b-8)')
#
#fig.suptitle('THEMIS-A', y=0.99)
#
#fig.subplots_adjust(hspace=0)
#fig.tight_layout(pad=0)
#
#if os.path.isdir(path_base_save_plot):
#    fig_path = os.path.join(path_base_save_plot, 'Figure_2b.png')
#    print(fig_path)
#    fig.savefig(fig_path)
#    plt.close(fig)
#else:
#    plt.show()
#    plt.close(fig)

# KAWの確認に適した時間窓$T_{\mathrm{window}}$の検討

```math
\frac{1}{v_{\mathrm{A}}} \frac{|\bf{E}_{\perp}|}{|\bf{B}_{\perp}|} = \frac{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2}}{\sqrt{1 + \frac{1}{2} \left( k_{\perp} \rho_{\mathrm{i}} \right)^{2} \left( 1 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)}} = \sqrt{10} \\
k_{\perp} \rho_{\mathrm{i}} = \frac{f_{\mathrm{sc}}}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \\
\therefore T_{\mathrm{window}} := \frac{1}{f_{\mathrm{sc}}} = \frac{1}{f_{\mathrm{ci}}} \frac{v_{\mathrm{thi}}}{V_{\mathrm{sys}\perp}} \left[ 9 + \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \left\{ 10 + \sqrt{117 \left( \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} \right)^{2} + 180 \frac{T_{\mathrm{e}}}{T_{\mathrm{i}}} + 100} \right\} \right]^{-\frac{1}{2}}
```

In [ ]:
def dedup_and_sort(ds):
    ds = ds.sortby("time")
    t = ds["time"].values
    _, keep = np.unique(t, return_index=True)  # 先勝ちで一意化
    return ds.isel(time=np.sort(keep))

ds_parameter_clean = dedup_and_sort(ds_parameter)
ds_velocity_ms_clean = dedup_and_sort(ds_velocity_ms)

ds_parameter_interp = ds_parameter_clean.interp(time=ds_velocity_ms_clean.time)
print(ds_parameter_interp)
print(ds_velocity_ms_clean)

In [ ]:
T_window = 1. / ds_parameter_interp['proton_cycl_freq_Hz'].data * ds_velocity_ms_clean['ion_thermal_speed'].data / ds_velocity_ms_clean['perp_sys_speed'].data / np.sqrt(9. + 1. / ds_parameter_interp['i-e_temp_ratio'].data * (10. + np.sqrt(117. / (ds_parameter_interp['i-e_temp_ratio'].data)**(2.) + 180. / ds_parameter_interp['i-e_temp_ratio'].data + 100.)))

da_T_window = xr.DataArray(data=T_window, dims=('time'), coords={'time': ds_velocity_ms_clean.time}, name='T_window')
da_T_window

In [ ]:
time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
da_T_window_analysis = da_T_window.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

mpl.rcParams['font.size'] = 25
fig = plt.figure(figsize=(10, 4))
ax = fig.add_subplot(111)
ax.plot(da_T_window_analysis.time, da_T_window_analysis.data, c='k', lw=1)
ax.minorticks_on()
ax.set_ylabel(r'$T_{\mathrm{window}}$' + '\n[sec]')
ax.set_yscale('log')
ax.set_ylim(ymin=1)
ax.grid(which='both', alpha=0.5)
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

fig.tight_layout()
plt.show(fig)
print(np.nanmin(da_T_window_analysis), np.nanmean(da_T_window_analysis))

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_{dt_time}sec'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------
dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:30:00', '2022-09-01T22:35:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_{dt_time}sec'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------
dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T22:47:30', '2022-09-01T22:52:30']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 12

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_{dt_time}sec'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------
dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig = psdptx.plot_freq_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)

time_range = ['2022-09-01T23:05:00', '2022-09-01T23:10:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

In [ ]:
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)
fig, fitres = psdptx.plot_k_spectrum_dual(data_dict, np.datetime64('2022-09-01T22:30:00'), 
                                   dt_sec=1, k_range=(1e-1,1e2), 
                                   n_bins=30, fit_range=(3,30))

if fig is not None:
    fig.show()
print(fitres)

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------
dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)
        return fit_results

time_range = ['2022-09-01T22:30:00', '2022-09-01T22:35:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------
dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)
        return fit_results

time_range = ['2022-09-01T22:47:30', '2022-09-01T22:52:30']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------
dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)
        return fit_results

#time_range = ['2022-09-01T23:05:00', '2022-09-01T23:10:00']
time_range = ['2022-09-01T22:30:45', '2022-09-01T22:31:45']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")

In [ ]:
import os
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
import matplotlib as mpl
import matplotlib.pyplot as plt

# --- モジュール読み込み ---
import sys, importlib
importlib.invalidate_caches()
os.chdir('..')
print(os.getcwd())
import module_handmade.psd_plotter_themis_xarray as psdptx
importlib.reload(psdptx)
os.chdir('./KAW_observation')
print(os.getcwd())

mpl.rcdefaults()
mpl.rcParams['font.size'] = 15

# ------------------------------------------------------------
# 0. 出力フォルダ
# ------------------------------------------------------------
dt_time = 1
out_dir = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/22-24_cleaned_CWT_{dt_time}sec_k_rhoi'
os.makedirs(out_dir, exist_ok=True)

# ------------------------------------------------------------
# 1. データ準備（ここは一度だけ）
# ------------------------------------------------------------
dsets_8_clean = [ds_8_fac_cwt_clean_seg0, ds_8_fac_cwt_clean_seg1, ds_8_fac_cwt_clean_seg2]
dsets_128     = [ds_128_fac_cwt_seg0,     ds_128_fac_cwt_seg1,     ds_128_fac_cwt_seg2]

data_dict = psdptx.build_data_dict_xr(
    dsets_8_clean, dsets_128, ds_velocity_ms, ds_parameter,
    f_split=4.0, cutoff_freq=[1/100, 0.7/3, 5/3, 64]
)

if data_dict is None:
    print("データ準備に失敗。終了。")
    raise SystemExit

# ------------------------------------------------------------
# 2. X秒ごとにプロットして保存（並列）
# ------------------------------------------------------------
def process_and_save_plot(t_start, dd, dt, out_dir):
    fig, fit_results = psdptx.plot_k_spectrum_dual(dd, np.datetime64(t_start), dt_sec=dt, n_bins=30)
    if fig is None:
        return
    try:
        fname = pd.Timestamp(t_start).strftime('%Y-%m-%dT%H%M%S.png')
        fig.savefig(os.path.join(out_dir, fname), dpi=200, bbox_inches='tight')
    finally:
        plt.close(fig)
        return fit_results

time_range = ['2022-09-01T22:25:00', '2022-09-01T23:25:00']
t_min, t_max = pd.to_datetime(time_range)
time_steps = pd.date_range(start=t_min, end=t_max, freq=f'{dt_time}s')

print(f"Processing {len(time_steps)} plots in parallel...")

results_list = Parallel(n_jobs=-1, backend="loky", verbose=10)(
    delayed(process_and_save_plot)(t, data_dict, dt_time, out_dir) for t in time_steps
)

print('Finished saving all plots!')

print("\n--- Saving and plotting fitting results ---")

# Noneが含まれる可能性を考慮してフィルタリング
results_list = [r for r in results_list if r is not None]

if results_list:
    # リストからDataFrameを作成
    df_results = pd.DataFrame(results_list)
    df_results = df_results.set_index('time').sort_index()

    # CSVファイルとして保存
    time_str_start = t_min.strftime('%Y%m%d_%H%M%S')
    time_str_end = t_max.strftime('%Y%m%d_%H%M%S')
    kappa_csv_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.csv'
    csv_path = os.path.join(out_dir, kappa_csv_filename)
    df_results.to_csv(csv_path)
    print(f"Fitting results saved to {csv_path}")

    S_para_window   = S_para.sel(time=slice(t_min, t_max))

    # --- 時間変化をプロット ---
    fig_kappa = plt.figure(figsize=(10, 6))
    gs = fig_kappa.add_gridspec(3, 1)
    ax_0 = fig_kappa.add_subplot(gs[0:2, 0])
    ax_1 = fig_kappa.add_subplot(gs[2, 0], sharex=ax_0)

    ax_0.tick_params(axis='x', which='both', labelbottom=False)
    
    # kappa_Bのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_B'], yerr=df_results['kappa_B_err'],
                fmt='o-', color='blue', label=r'$\kappa_{\mathrm{B}}$', ms=2, elinewidth=0.5, capsize=2)
    
    # kappa_Eのプロット（エラーバー付き）
    ax_0.errorbar(df_results.index, df_results['kappa_E'], yerr=df_results['kappa_E_err'],
                fmt='o-', color='orange', label=r'$\kappa_{\mathrm{E}}$', ms=2, elinewidth=0.5, capsize=2)
    
    ax_0.set_title(r'Time evolution of spectral index $\kappa$ (THEMIS-A)')
    ax_0.set_ylabel(r'Spectral index $\kappa$')
    ax_0.legend()
    ax_0.minorticks_on()
    ax_0.grid(True, linestyle=':', which='both')

    ax_1.plot(S_para_window.time, S_para_window.data*1E3, c='k', linewidth=0.5)
    ax_1.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
    ax_1.minorticks_on()
    ax_1.grid(True, linestyle=':', which='both')
    ax_1.set_xlabel('Time')

    ax_0.set_xlim(t_min, t_max)

    fig_kappa.tight_layout()

    # プロットを画像として保存
    kappa_plot_filename = f'kappa_timeseries_{time_str_start}_to_{time_str_end}.png'
    kappa_plot_path = os.path.join(out_dir, kappa_plot_filename)
    fig_kappa.savefig(kappa_plot_path, dpi=200)
    plt.close(fig_kappa)
    print(f"Time series plot of kappa saved to {kappa_plot_path}")

else:
    print("No fitting results to save or plot.")